In [ ]:
import os
import glob
import torch
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold

# 1. Setup Directories and Groups (matching your extraction notebook)
data_dir = os.path.expanduser('~/Desktop/FINAL_brainmath_cleaned_nifti')
all_files = glob.glob(os.path.join(data_dir, '*_cleaned3D.nii.gz'))

mld_subs = ['059', '065', '067', '069', '071', '075', '076', '077', '078', '083', '088', '095', '096', '103', '106']
td_subs = ['090', '036', '013', '008', '057', '070', '023', '024', '053', '044', '034', '060', '007', '027', '010']

# 2. Initialize empty lists
images = []
labels_list = []
groups = []  # Critical for preventing data leakage

# 3. Parse the directory to populate the lists
for filepath in all_files:
    filename = os.path.basename(filepath)
    
    # Extract subject ID for grouping and labeling
    parts = filename.split('_')
    sub_part = [p for p in parts if p.startswith('sub-')]
    if not sub_part:
        continue
    sub_id = sub_part[0].split('-')[1]
    
    # Assign label: 1 for MLD, 0 for TD
    if sub_id in mld_subs:
        label = 1
    elif sub_id in td_subs:
        label = 0
    else:
        continue # Skip subjects not in either list
        
    images.append(filepath)
    labels_list.append(label)
    groups.append(sub_id) # Save the subject ID as the "group"

# print out the lists for checking
print(f"Images: {len(images)}")
print(f"Labels: {len(labels_list)}")
print(f"Groups: {len(groups)}")

labels_array = np.array(labels_list)
images_array = np.array(images)
groups_array = np.array(groups)
# Convert labels to one-hot format for MONAI DenseNet/ResNet
labels_tensor = torch.nn.functional.one_hot(torch.as_tensor(labels_array)).float()

# print(f"Total scans loaded: {len(images)}")

Images: 240
Labels: 240
Groups: 240


In [ ]:
import os
import numpy as np
import json
from sklearn.model_selection import StratifiedGroupKFold

# [Assume images_array, labels_array, and groups_array are already loaded]

n_splits = 5
sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42)

# Create a dictionary to hold all our fold data
all_folds_data = {}

for fold, (train_idx, val_idx) in enumerate(sgkf.split(images_array, labels_array, groups_array)):
    
    # Store the exact filenames and labels for this fold
    # .tolist() is used because JSON cannot save numpy arrays natively
    all_folds_data[f"fold_{fold + 1}"] = {
        "train_images": images_array[train_idx].tolist(),
        "train_labels": labels_array[train_idx].tolist(),
        "val_images": images_array[val_idx].tolist(),
        "val_labels": labels_array[val_idx].tolist()
    }

# Save the dictionary to a JSON file on your disk
save_path = os.path.join(os.path.expanduser('~/Desktop'), 'brain-math/deeplearn/kfold_splits_mean.json')
with open(save_path, 'w') as f:
    json.dump(all_folds_data, f, indent=4)

print(f"Successfully saved all 5 folds to: {save_path}")

Successfully saved all 5 folds to: /Users/jchong058/Desktop/brain-math/monai/folds/kfold_splits.json
